In [1]:
import sys
import torch
import transformers
import datasets
import pandas as pd

print("=" * 70)
print("TEXT SUMMARIZATION PROJECT - ENVIRONMENT AUDIT")
print("=" * 70)

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"Pandas:       {pd.__version__}")

print("\n" + "=" * 70)
print("GPU INFORMATION")
print("=" * 70)

print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:  {torch.version.cuda}")
    print(f"GPU memory:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected.")

TEXT SUMMARIZATION PROJECT - ENVIRONMENT AUDIT
Python:       3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch:      2.11.0+cu128
Transformers: 5.16.1
Datasets:     4.8.5
Pandas:       2.2.3

GPU INFORMATION
CUDA available: True
GPU:           Tesla T4
CUDA version:  12.8
GPU memory:    14.56 GB


In [2]:
!pip install -q evaluate rouge_score sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [3]:
import evaluate
import rouge_score
import transformers
import datasets

print("=" * 70)
print("LIBRARY VERIFICATION")
print("=" * 70)

print(f"Evaluate:      {evaluate.__version__}")
print(f"Transformers:  {transformers.__version__}")
print(f"Datasets:      {datasets.__version__}")
print("ROUGE:         Available")

print("\nAll required libraries are ready.")

LIBRARY VERIFICATION
Evaluate:      0.4.6
Transformers:  5.16.1
Datasets:      4.8.5
ROUGE:         Available

All required libraries are ready.


In [5]:
from datasets import load_dataset

print("=" * 70)
print("LOADING CNN/DAILYMAIL DATASET")
print("=" * 70)

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

print("\nDataset loaded successfully.")
print(dataset)

print("\n" + "=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

for split in dataset:
    print(f"{split}: {len(dataset[split]):,} examples")

print("\nColumns:")
print(dataset["train"].column_names)

print("\n" + "=" * 70)
print("FIRST EXAMPLE")
print("=" * 70)

sample = dataset["train"][0]

print(f"\nArticle length: {len(sample['article']):,} characters")
print(f"Summary length: {len(sample['highlights']):,} characters")

print("\nArticle:")
print(sample["article"][:1000])

print("\nReference Summary:")
print(sample["highlights"])

LOADING CNN/DAILYMAIL DATASET


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]


Dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

DATASET STRUCTURE
train: 287,113 examples
validation: 13,368 examples
test: 11,490 examples

Columns:
['article', 'highlights', 'id']

FIRST EXAMPLE

Article length: 2,527 characters
Summary length: 217 characters

Article:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fas

In [6]:
import numpy as np

print("=" * 70)
print("CNN/DAILYMAIL TEXT LENGTH ANALYSIS")
print("=" * 70)

# Analyze a representative sample from the training set
analysis_sample = dataset["train"].select(range(10_000))

article_char_lengths = np.array([
    len(text) for text in analysis_sample["article"]
])

summary_char_lengths = np.array([
    len(text) for text in analysis_sample["highlights"]
])

print("\n" + "=" * 70)
print("ARTICLE CHARACTER LENGTHS")
print("=" * 70)

print(f"Minimum:  {article_char_lengths.min():,}")
print(f"Mean:     {article_char_lengths.mean():,.0f}")
print(f"Median:   {np.median(article_char_lengths):,.0f}")
print(f"95th pct:  {np.percentile(article_char_lengths, 95):,.0f}")
print(f"99th pct:  {np.percentile(article_char_lengths, 99):,.0f}")
print(f"Maximum:  {article_char_lengths.max():,}")

print("\n" + "=" * 70)
print("SUMMARY CHARACTER LENGTHS")
print("=" * 70)

print(f"Minimum:  {summary_char_lengths.min():,}")
print(f"Mean:     {summary_char_lengths.mean():,.0f}")
print(f"Median:   {np.median(summary_char_lengths):,.0f}")
print(f"95th pct:  {np.percentile(summary_char_lengths, 95):,.0f}")
print(f"99th pct:  {np.percentile(summary_char_lengths, 99):,.0f}")
print(f"Maximum:  {summary_char_lengths.max():,}")

CNN/DAILYMAIL TEXT LENGTH ANALYSIS

ARTICLE CHARACTER LENGTHS
Minimum:  106
Mean:     3,695
Median:   3,444
95th pct:  6,981
99th pct:  8,766
Maximum:  11,265

SUMMARY CHARACTER LENGTHS
Minimum:  65
Mean:     264
Median:   271
95th pct:  327
99th pct:  350
Maximum:  507


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("=" * 70)
print("BART TOKENIZER")
print("=" * 70)

print(f"Model: {MODEL_NAME}")
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

BART TOKENIZER
Model: facebook/bart-base
Tokenizer: RobertaTokenizer
Vocabulary size: 50265
Model max length: 1000000000000000019884624838656


In [8]:
print("=" * 70)
print("BART TOKEN LENGTH ANALYSIS")
print("=" * 70)

article_token_lengths = []
summary_token_lengths = []

for article, summary in zip(
    analysis_sample["article"],
    analysis_sample["highlights"]
):
    article_tokens = tokenizer(
        article,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    summary_tokens = tokenizer(
        summary,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    article_token_lengths.append(len(article_tokens))
    summary_token_lengths.append(len(summary_tokens))

article_token_lengths = np.array(article_token_lengths)
summary_token_lengths = np.array(summary_token_lengths)

print("\n" + "=" * 70)
print("ARTICLE TOKEN LENGTHS")
print("=" * 70)

print(f"Minimum:  {article_token_lengths.min():,}")
print(f"Mean:     {article_token_lengths.mean():,.0f}")
print(f"Median:   {np.median(article_token_lengths):,.0f}")
print(f"95th pct:  {np.percentile(article_token_lengths, 95):,.0f}")
print(f"99th pct:  {np.percentile(article_token_lengths, 99):,.0f}")
print(f"Maximum:  {article_token_lengths.max():,}")

print("\n" + "=" * 70)
print("SUMMARY TOKEN LENGTHS")
print("=" * 70)

print(f"Minimum:  {summary_token_lengths.min():,}")
print(f"Mean:     {summary_token_lengths.mean():,.0f}")
print(f"Median:   {np.median(summary_token_lengths):,.0f}")
print(f"95th pct:  {np.percentile(summary_token_lengths, 95):,.0f}")
print(f"99th pct:  {np.percentile(summary_token_lengths, 99):,.0f}")
print(f"Maximum:  {summary_token_lengths.max():,}")

BART TOKEN LENGTH ANALYSIS

ARTICLE TOKEN LENGTHS
Minimum:  27
Mean:     784
Median:   726
95th pct:  1,499
99th pct:  1,902
Maximum:  2,622

SUMMARY TOKEN LENGTHS
Minimum:  14
Mean:     60
Median:   61
95th pct:  78
99th pct:  84
Maximum:  113


In [9]:
# Model configuration
MODEL_NAME = "facebook/bart-base"

# Tokenization limits
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)

print(f"Model:             {MODEL_NAME}")
print(f"Max input length:  {MAX_INPUT_LENGTH}")
print(f"Max target length: {MAX_TARGET_LENGTH}")
print(f"Device:            {'cuda' if torch.cuda.is_available() else 'cpu'}")

PROJECT CONFIGURATION
Model:             facebook/bart-base
Max input length:  1024
Max target length: 128
Device:            cuda


In [10]:
TRAIN_SIZE = 8_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000

train_dataset = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
val_dataset = dataset["validation"].shuffle(seed=42).select(range(VAL_SIZE))
test_dataset = dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

print("=" * 70)
print("PROJECT DATASET SUBSETS")
print("=" * 70)

print(f"Train:       {len(train_dataset):,}")
print(f"Validation:  {len(val_dataset):,}")
print(f"Test:        {len(test_dataset):,}")

print("\nRandom seed: 42")

PROJECT DATASET SUBSETS
Train:       8,000
Validation:  1,000
Test:        1,000

Random seed: 42


In [11]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["article"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


test_tokenized = train_dataset.select(range(3)).map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

print("=" * 70)
print("TOKENIZATION TEST")
print("=" * 70)

for i in range(3):
    print(f"\nExample {i + 1}")
    print(f"Input tokens:  {len(test_tokenized[i]['input_ids'])}")
    print(f"Label tokens:  {len(test_tokenized[i]['labels'])}")
    print(f"Attention mask: {len(test_tokenized[i]['attention_mask'])}")

print("\nTokenization test completed successfully.")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

TOKENIZATION TEST

Example 1
Input tokens:  663
Label tokens:  47
Attention mask: 663

Example 2
Input tokens:  1024
Label tokens:  68
Attention mask: 1024

Example 3
Input tokens:  741
Label tokens:  66
Attention mask: 741

Tokenization test completed successfully.


In [12]:
print("=" * 70)
print("TOKENIZING DATASETS")
print("=" * 70)

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    num_proc=2,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test data"
)

print("\n" + "=" * 70)
print("TOKENIZATION COMPLETED")
print("=" * 70)

print(f"Train examples:      {len(tokenized_train):,}")
print(f"Validation examples: {len(tokenized_val):,}")
print(f"Test examples:       {len(tokenized_test):,}")

print("\nFeatures:")
print(tokenized_train.column_names)

TOKENIZING DATASETS


Tokenizing training data (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing validation data (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing test data (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]


TOKENIZATION COMPLETED
Train examples:      8,000
Validation examples: 1,000
Test examples:       1,000

Features:
['input_ids', 'attention_mask', 'labels']


In [13]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=MODEL_NAME,
    padding=True
)

print("=" * 70)
print("DATA COLLATOR")
print("=" * 70)

print("Data collator created successfully.")
print(f"Type: {data_collator.__class__.__name__}")

DATA COLLATOR
Data collator created successfully.
Type: DataCollatorForSeq2Seq


In [14]:
from transformers import AutoModelForSeq2SeqLM

print("=" * 70)
print("LOADING BART MODEL")
print("=" * 70)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("\nModel loaded successfully.")
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size:           {total_params * 4 / (1024**2):.2f} MB (FP32 estimate)")

LOADING BART MODEL


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]


Model loaded successfully.
Model: facebook/bart-base
Device: cuda

MODEL PARAMETERS
Total parameters:     139,420,416
Trainable parameters: 139,420,416
Model size:           531.85 MB (FP32 estimate)


In [16]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_cnn_dailymail",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimization
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,

    # Memory optimization
    fp16=True,
    gradient_checkpointing=True,

    # Evaluation & checkpoints
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # Logging
    logging_strategy="steps",
    logging_steps=100,

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reproducibility
    seed=42,

    # Reporting
    report_to="none"
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(f"Epochs:                    {training_args.num_train_epochs}")
print(f"Train batch size/GPU:     {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation:    {training_args.gradient_accumulation_steps}")
print(f"Effective batch size:     {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate:             {training_args.learning_rate}")
print(f"Warmup steps:              {training_args.warmup_steps}")
print(f"FP16:                      {training_args.fp16}")
print(f"Gradient checkpointing:    {training_args.gradient_checkpointing}")
print(f"Evaluation steps:          {training_args.eval_steps}")
print(f"Save steps:                {training_args.save_steps}")

TRAINING CONFIGURATION
Epochs:                    2
Train batch size/GPU:     2
Gradient accumulation:    8
Effective batch size:     16
Learning rate:             5e-05
Warmup steps:              100
FP16:                      True
Gradient checkpointing:    True
Evaluation steps:          500
Save steps:                500


In [17]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("=" * 70)
print("TRAINER CREATED")
print("=" * 70)

print(f"Training examples:   {len(trainer.train_dataset):,}")
print(f"Validation examples: {len(trainer.eval_dataset):,}")

print("\n" + "=" * 70)
print("GPU MEMORY BEFORE BATCH TEST")
print("=" * 70)

if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

print("\nTrainer is ready.")

TRAINER CREATED
Training examples:   8,000
Validation examples: 1,000

GPU MEMORY BEFORE BATCH TEST
Allocated: 0.52 GB
Reserved:  0.58 GB

Trainer is ready.


In [18]:
import torch

print("=" * 70)
print("GPU BATCH SANITY CHECK")
print("=" * 70)

model.train()

batch = data_collator([
    tokenized_train[i]
    for i in range(2)
])

batch = {
    k: v.to(device)
    for k, v in batch.items()
}

print(f"Input shape:  {batch['input_ids'].shape}")
print(f"Labels shape: {batch['labels'].shape}")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.amp.autocast("cuda", dtype=torch.float16):
    outputs = model(**batch)

loss = outputs.loss
loss.backward()

peak_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"\nLoss: {loss.item():.4f}")
print(f"Peak GPU memory: {peak_memory:.2f} GB")

del outputs, loss, batch
torch.cuda.empty_cache()

print("\nBatch sanity check completed successfully.")

GPU BATCH SANITY CHECK
Input shape:  torch.Size([2, 1024])
Labels shape: torch.Size([2, 68])

Loss: 5.1809
Peak GPU memory: 1.63 GB

Batch sanity check completed successfully.


In [19]:
print("=" * 70)
print("FINAL TRAINING PREPARATION")
print("=" * 70)

model.config.use_cache = False

print(f"Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"Use cache:              {model.config.use_cache}")
print(f"FP16:                   {training_args.fp16}")
print(f"Epochs:                 {training_args.num_train_epochs}")
print(f"Train examples:         {len(tokenized_train):,}")
print(f"Validation examples:    {len(tokenized_val):,}")

print("\nModel is ready for fine-tuning.")

FINAL TRAINING PREPARATION
Gradient checkpointing: True
Use cache:              False
FP16:                   True
Epochs:                 2
Train examples:         8,000
Validation examples:    1,000

Model is ready for fine-tuning.


In [20]:
print("=" * 70)
print("STARTING BART FINE-TUNING")
print("=" * 70)

train_result = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training steps: {train_result.global_step}")

STARTING BART FINE-TUNING


Step,Training Loss,Validation Loss
500,17.225300,1.794385
1000,15.365170,1.757423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



TRAINING COMPLETED
Training loss: 17.2732
Training steps: 1000


In [21]:
import os
import json

print("=" * 70)
print("TRAINING ARTIFACT VALIDATION")
print("=" * 70)

# 1. Trainer state
print("\n[1] Trainer State")
print(f"Global step: {trainer.state.global_step}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best metric: {trainer.state.best_metric}")

# 2. Training arguments
print("\n[2] Best Model Configuration")
print(f"Load best model at end: {trainer.args.load_best_model_at_end}")
print(f"Metric for best model: {trainer.args.metric_for_best_model}")
print(f"Greater is better: {trainer.args.greater_is_better}")

# 3. Log history
print("\n[3] Evaluation History")

for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(
            f"Step {entry.get('step')}: "
            f"eval_loss={entry['eval_loss']:.6f}"
        )

# 4. Check output directory
print("\n[4] Output Directory")

if os.path.exists(trainer.args.output_dir):
    items = sorted(os.listdir(trainer.args.output_dir))

    for item in items:
        path = os.path.join(trainer.args.output_dir, item)

        if os.path.isdir(path):
            print(f"[DIR]  {item}")
        else:
            size_mb = os.path.getsize(path) / (1024 ** 2)
            print(f"[FILE] {item} - {size_mb:.2f} MB")
else:
    print("Output directory does not exist.")

# 5. Checkpoint directories
print("\n[5] Checkpoints")

checkpoints = []

if os.path.exists(trainer.args.output_dir):
    for item in os.listdir(trainer.args.output_dir):
        if item.startswith("checkpoint-"):
            path = os.path.join(trainer.args.output_dir, item)

            if os.path.isdir(path):
                checkpoints.append(item)

for checkpoint in sorted(checkpoints, key=lambda x: int(x.split("-")[1])):
    checkpoint_path = os.path.join(trainer.args.output_dir, checkpoint)

    print(f"\n{checkpoint}")

    for item in sorted(os.listdir(checkpoint_path)):
        path = os.path.join(checkpoint_path, item)

        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / (1024 ** 2)
            print(f"  {item} - {size_mb:.2f} MB")

print("\n" + "=" * 70)
print("VALIDATION INSPECTION COMPLETED")
print("=" * 70)

TRAINING ARTIFACT VALIDATION

[1] Trainer State
Global step: 1000
Best checkpoint: ./bart_cnn_dailymail/checkpoint-1000
Best metric: 1.7574230432510376

[2] Best Model Configuration
Load best model at end: True
Metric for best model: eval_loss
Greater is better: False

[3] Evaluation History
Step 500: eval_loss=1.794385
Step 1000: eval_loss=1.757423

[4] Output Directory
[DIR]  checkpoint-1000
[DIR]  checkpoint-500

[5] Checkpoints

checkpoint-500
  config.json - 0.00 MB
  generation_config.json - 0.00 MB
  model.safetensors - 532.07 MB
  optimizer.pt - 1063.90 MB
  rng_state.pth - 0.01 MB
  scaler.pt - 0.00 MB
  scheduler.pt - 0.00 MB
  tokenizer.json - 3.39 MB
  tokenizer_config.json - 0.00 MB
  trainer_state.json - 0.00 MB
  training_args.bin - 0.01 MB

checkpoint-1000
  config.json - 0.00 MB
  generation_config.json - 0.00 MB
  model.safetensors - 532.07 MB
  optimizer.pt - 1063.90 MB
  rng_state.pth - 0.01 MB
  scaler.pt - 0.00 MB
  scheduler.pt - 0.00 MB
  tokenizer.json - 3.39 M

In [23]:
print("=" * 70)
print("TEST DATASET AVAILABILITY CHECK")
print("=" * 70)

# Check whether the original test dataset exists
print("\n[1] Original test dataset")
print(f"'test' exists: {'test' in globals()}")

if "test" in globals():
    print(f"Type: {type(test)}")
    print(f"Number of examples: {len(test)}")
    print(f"Columns: {test.column_names}")

# Check whether the tokenized test dataset exists
print("\n[2] Tokenized test dataset")
print(f"'tokenized_test' exists: {'tokenized_test' in globals()}")

if "tokenized_test" in globals():
    print(f"Type: {type(tokenized_test)}")
    print(f"Number of examples: {len(tokenized_test)}")
    print(f"Columns: {tokenized_test.column_names}")

print("\n" + "=" * 70)
print("CHECK COMPLETED")
print("=" * 70)

TEST DATASET AVAILABILITY CHECK

[1] Original test dataset
'test' exists: False

[2] Tokenized test dataset
'tokenized_test' exists: True
Type: <class 'datasets.arrow_dataset.Dataset'>
Number of examples: 1000
Columns: ['input_ids', 'attention_mask', 'labels']

CHECK COMPLETED


In [24]:
import torch

print("=" * 70)
print("FINE-TUNED MODEL - REAL INFERENCE TEST")
print("=" * 70)

model = trainer.model
model.eval()

device = next(model.parameters()).device

print(f"\nModel device: {device}")
print(f"Test examples available: {len(tokenized_test)}")

num_examples = 3

for i in range(num_examples):
    example = tokenized_test[i]

    # Decode the real article from the tokenized test set
    article = tokenizer.decode(
        example["input_ids"],
        skip_special_tokens=True
    )

    # Decode the real reference summary
    reference = tokenizer.decode(
        example["labels"],
        skip_special_tokens=True
    )

    # Prepare model input
    input_ids = torch.tensor(
        [example["input_ids"]],
        dtype=torch.long,
        device=device
    )

    attention_mask = torch.tensor(
        [example["attention_mask"]],
        dtype=torch.long,
        device=device
    )

    # Generate summary
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    print("\n" + "=" * 70)
    print(f"EXAMPLE {i + 1}")
    print("=" * 70)

    print("\nARTICLE:")
    print(article[:2000])

    print("\nREFERENCE SUMMARY:")
    print(reference)

    print("\nMODEL SUMMARY:")
    print(generated_summary)

print("\n" + "=" * 70)
print("INFERENCE TEST COMPLETED")
print("=" * 70)

FINE-TUNED MODEL - REAL INFERENCE TEST

Model device: cuda:0
Test examples available: 1000

EXAMPLE 1

ARTICLE:
(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I see it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I see it in the new surgeon general who cites data showing just how helpful it can be. I see a revolution in the attitudes of everyday Americans. For the first time a majority, 53%, favor its legalization, with 77% supporting it for medical purposes. Support for legalization has risen 11 points in the past few years alone. In 1969, the first time Pew asked the question about legalization, only 12% of the nation was in favor. I see a re

In [25]:
import evaluate
import transformers
import torch

print("=" * 70)
print("ROUGE EVALUATION ENVIRONMENT CHECK")
print("=" * 70)

print(f"Transformers version: {transformers.__version__}")
print(f"Evaluate version: {evaluate.__version__}")
print(f"PyTorch version: {torch.__version__}")

print(f"\nGPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory allocated: "
        f"{torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB"
    )

print("\nLoading ROUGE metric...")

rouge = evaluate.load("rouge")

print("ROUGE metric loaded successfully.")

print("\nAvailable ROUGE metrics:")
print(rouge.features)

print("\n" + "=" * 70)
print("ENVIRONMENT CHECK COMPLETED")
print("=" * 70)

ROUGE EVALUATION ENVIRONMENT CHECK
Transformers version: 5.16.1
Evaluate version: 0.4.6
PyTorch version: 2.11.0+cu128

GPU available: True
GPU: Tesla T4
GPU memory allocated: 1.58 GB

Loading ROUGE metric...


ROUGE metric loaded successfully.

Available ROUGE metrics:
[{'predictions': Value('string'), 'references': List(Value('string'))}, {'predictions': Value('string'), 'references': Value('string')}]

ENVIRONMENT CHECK COMPLETED


In [28]:
import torch
from tqdm.auto import tqdm

print("=" * 70)
print("GENERATING SUMMARIES FOR ROUGE EVALUATION")
print("DYNAMIC PADDING + LABEL DECODING FIX")
print("=" * 70)

model = trainer.model
model.eval()

device = next(model.parameters()).device

# Evaluation configuration
BATCH_SIZE = 4
NUM_BEAMS = 4
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128

predictions = []
references = []

print(f"\nDevice: {device}")
print(f"Test examples: {len(tokenized_test)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Beam size: {NUM_BEAMS}")
print(f"Max input length: {MAX_INPUT_LENGTH}")
print(f"Max output length: {MAX_TARGET_LENGTH}")

for start_idx in tqdm(
    range(0, len(tokenized_test), BATCH_SIZE),
    desc="Generating summaries"
):
    end_idx = min(start_idx + BATCH_SIZE, len(tokenized_test))

    examples = [
        tokenized_test[i]
        for i in range(start_idx, end_idx)
    ]

    # Dynamically pad the batch
    batch = data_collator(examples)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_TARGET_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True
        )

    # Decode generated summaries
    batch_predictions = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    # Replace -100 padding values before decoding references
    labels = batch["labels"].clone()
    labels[labels == -100] = tokenizer.pad_token_id

    batch_references = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    predictions.extend(batch_predictions)
    references.extend(batch_references)

print("\n" + "=" * 70)
print("GENERATION COMPLETED")
print("=" * 70)

print(f"Generated predictions: {len(predictions)}")
print(f"Collected references: {len(references)}")

print("\nFirst prediction:")
print(predictions[0])

print("\nFirst reference:")
print(references[0])

GENERATING SUMMARIES FOR ROUGE EVALUATION
DYNAMIC PADDING + LABEL DECODING FIX

Device: cuda:0
Test examples: 1000
Batch size: 4
Beam size: 4
Max input length: 1024
Max output length: 128


Generating summaries:   0%|          | 0/250 [00:00<?, ?it/s]


GENERATION COMPLETED
Generated predictions: 1000
Collected references: 1000

First prediction:
The revolution in attitudes of everyday Americans has risen 11 points in the past few years alone .
I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world .
The revolution is burning white hot among young people, but also among parents and grandparents .

First reference:
CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"


In [29]:
print("=" * 70)
print("CALCULATING ROUGE SCORES")
print("=" * 70)

# Calculate ROUGE on the 1,000 generated test summaries
rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("\nROUGE RESULTS")
print("-" * 70)

for metric, score in rouge_results.items():
    print(f"{metric}: {score:.6f}")

print("\n" + "=" * 70)
print("ROUGE EVALUATION COMPLETED")
print("=" * 70)

print(f"Number of predictions: {len(predictions)}")
print(f"Number of references: {len(references)}")

CALCULATING ROUGE SCORES

ROUGE RESULTS
----------------------------------------------------------------------
rouge1: 0.406423
rouge2: 0.180911
rougeL: 0.276005
rougeLsum: 0.374562

ROUGE EVALUATION COMPLETED
Number of predictions: 1000
Number of references: 1000


In [30]:
print("=" * 70)
print("LOADING PRETRAINED BART-BASE BASELINE")
print("=" * 70)

import torch
from transformers import AutoModelForSeq2SeqLM

# Load a fresh copy of the original pretrained BART-base model.
# This model has NOT been fine-tuned on CNN/DailyMail.
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move model to GPU if available.
baseline_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model = baseline_model.to(baseline_device)
baseline_model.eval()

print("\nBASELINE MODEL READY")
print("-" * 70)
print(f"Model name: {MODEL_NAME}")
print(f"Device: {baseline_device}")
print(f"Parameters: {sum(p.numel() for p in baseline_model.parameters()):,}")
print(f"Evaluation mode: {not baseline_model.training}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

print("\nImportant: This is the original pretrained BART-base model,")
print("before fine-tuning on the CNN/DailyMail dataset.")

print("\n" + "=" * 70)
print("BASELINE MODEL LOADING COMPLETED")
print("=" * 70)

LOADING PRETRAINED BART-BASE BASELINE


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]


BASELINE MODEL READY
----------------------------------------------------------------------
Model name: facebook/bart-base
Device: cuda
Parameters: 139,420,416
Evaluation mode: True
GPU: Tesla T4
GPU memory allocated: 2.11 GB

Important: This is the original pretrained BART-base model,
before fine-tuning on the CNN/DailyMail dataset.

BASELINE MODEL LOADING COMPLETED


In [31]:
print("=" * 70)
print("GENERATING BASELINE SUMMARIES")
print("=" * 70)

from tqdm.auto import tqdm

baseline_predictions = []

# Generate summaries for the same 1,000 test examples.
for start_idx in tqdm(
    range(0, len(tokenized_test), BATCH_SIZE),
    desc="Baseline generation"
):
    batch = tokenized_test[start_idx:start_idx + BATCH_SIZE]

    # Use the same dynamic padding strategy used during fine-tuned evaluation.
    batch_inputs = data_collator(
        [
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask
            }
            for input_ids, attention_mask in zip(
                batch["input_ids"],
                batch["attention_mask"]
            )
        ]
    )

    input_ids = batch_inputs["input_ids"].to(baseline_device)
    attention_mask = batch_inputs["attention_mask"].to(baseline_device)

    with torch.no_grad():
        generated_ids = baseline_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    baseline_predictions.extend(decoded)

print("\n" + "=" * 70)
print("BASELINE GENERATION COMPLETED")
print("=" * 70)

print(f"Generated baseline predictions: {len(baseline_predictions)}")

print("\nFirst baseline prediction:")
print("-" * 70)
print(baseline_predictions[0])

print("\nCorresponding reference:")
print("-" * 70)
print(references[0])

GENERATING BASELINE SUMMARIES


Baseline generation:   0%|          | 0/250 [00:00<?, ?it/s]


BASELINE GENERATION COMPLETED
Generated baseline predictions: 1000

First baseline prediction:
----------------------------------------------------------------------
(CNN) I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country. I saw it in politicians who once preferred to play it safe with this explosive issue but are now willing to stake their political futures on it. I have seen the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world, who are diving in head first. I am in the hands of the new surgeon general who cites data showing just how helpful it can be. I'm in the attitudes of everyday Americans.

Corresponding reference:
----------------------------------------------------------------------
CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for 

In [32]:
print("=" * 70)
print("CALCULATING BASELINE ROUGE SCORES")
print("=" * 70)

# Calculate ROUGE using the same test references and evaluation settings
# used for the fine-tuned model.
baseline_rouge_results = rouge.compute(
    predictions=baseline_predictions,
    references=references,
    use_stemmer=True
)

print("\nBASELINE ROUGE RESULTS")
print("-" * 70)

for metric, score in baseline_rouge_results.items():
    print(f"{metric}: {score:.6f}")

print("\n" + "=" * 70)
print("BASELINE ROUGE EVALUATION COMPLETED")
print("=" * 70)

print(f"Number of baseline predictions: {len(baseline_predictions)}")
print(f"Number of references: {len(references)}")

CALCULATING BASELINE ROUGE SCORES

BASELINE ROUGE RESULTS
----------------------------------------------------------------------
rouge1: 0.392200
rouge2: 0.176266
rougeL: 0.245521
rougeLsum: 0.319560

BASELINE ROUGE EVALUATION COMPLETED
Number of baseline predictions: 1000
Number of references: 1000


In [33]:
print("=" * 70)
print("BASELINE VS FINE-TUNED ROUGE COMPARISON")
print("=" * 70)

fine_tuned_rouge_results = {
    "rouge1": 0.406423,
    "rouge2": 0.180911,
    "rougeL": 0.276005,
    "rougeLsum": 0.374562
}

comparison = {}

for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    baseline = baseline_rouge_results[metric]
    fine_tuned = fine_tuned_rouge_results[metric]

    absolute_improvement = fine_tuned - baseline
    relative_improvement = (absolute_improvement / baseline) * 100

    comparison[metric] = {
        "baseline": baseline,
        "fine_tuned": fine_tuned,
        "absolute_improvement": absolute_improvement,
        "relative_improvement_percent": relative_improvement
    }

print("\nMetric Comparison:")
print("-" * 70)

for metric, values in comparison.items():
    print(f"\n{metric.upper()}")
    print(f"Baseline:              {values['baseline']:.6f}")
    print(f"Fine-tuned:            {values['fine_tuned']:.6f}")
    print(f"Absolute improvement:  {values['absolute_improvement']:+.6f}")
    print(f"Relative improvement:  {values['relative_improvement_percent']:+.2f}%")

print("\n" + "=" * 70)
print("COMPARISON COMPLETED")
print("=" * 70)

BASELINE VS FINE-TUNED ROUGE COMPARISON

Metric Comparison:
----------------------------------------------------------------------

ROUGE1
Baseline:              0.392200
Fine-tuned:            0.406423
Absolute improvement:  +0.014223
Relative improvement:  +3.63%

ROUGE2
Baseline:              0.176266
Fine-tuned:            0.180911
Absolute improvement:  +0.004645
Relative improvement:  +2.64%

ROUGEL
Baseline:              0.245521
Fine-tuned:            0.276005
Absolute improvement:  +0.030484
Relative improvement:  +12.42%

ROUGELSUM
Baseline:              0.319560
Fine-tuned:            0.374562
Absolute improvement:  +0.055002
Relative improvement:  +17.21%

COMPARISON COMPLETED


In [34]:
import os
import torch
from safetensors.torch import load_file

print("=" * 70)
print("CHECKPOINT WEIGHT INTEGRITY VALIDATION")
print("=" * 70)

checkpoint_path = "./bart_cnn_dailymail/checkpoint-1000"
weights_path = os.path.join(checkpoint_path, "model.safetensors")

print(f"\nCheckpoint path: {checkpoint_path}")
print(f"Weights file exists: {os.path.exists(weights_path)}")

if not os.path.exists(weights_path):
    raise FileNotFoundError(
        f"Expected checkpoint weights were not found: {weights_path}"
    )

# Load checkpoint state dictionary without modifying the current model.
checkpoint_state = load_file(weights_path)

print(f"State dict tensors: {len(checkpoint_state):,}")

# Key tensors mentioned in the training warning.
key_tensors = [
    "model.encoder.embed_tokens.weight",
    "model.decoder.embed_tokens.weight",
    "lm_head.weight"
]

print("\n" + "=" * 70)
print("KEY WEIGHT VALIDATION")
print("=" * 70)

for key in key_tensors:
    exists = key in checkpoint_state
    print(f"\n{key}")
    print(f"Present in checkpoint: {exists}")

    if exists:
        print(f"Shape: {tuple(checkpoint_state[key].shape)}")

# Compare checkpoint tensor names with the current fine-tuned model.
current_state = trainer.model.state_dict()

checkpoint_keys = set(checkpoint_state.keys())
current_keys = set(current_state.keys())

missing_from_checkpoint = sorted(current_keys - checkpoint_keys)
extra_in_checkpoint = sorted(checkpoint_keys - current_keys)

print("\n" + "=" * 70)
print("STATE DICT KEY COMPARISON")
print("=" * 70)

print(f"Current model tensors:       {len(current_keys):,}")
print(f"Checkpoint tensors:          {len(checkpoint_keys):,}")
print(f"Missing from checkpoint:     {len(missing_from_checkpoint):,}")
print(f"Extra checkpoint tensors:    {len(extra_in_checkpoint):,}")

if missing_from_checkpoint:
    print("\nMissing keys:")
    for key in missing_from_checkpoint[:20]:
        print(f"  {key}")

if extra_in_checkpoint:
    print("\nExtra checkpoint keys:")
    for key in extra_in_checkpoint[:20]:
        print(f"  {key}")

# Verify shapes for keys shared by both state dictionaries.
shared_keys = checkpoint_keys & current_keys
shape_mismatches = []

for key in shared_keys:
    if checkpoint_state[key].shape != current_state[key].shape:
        shape_mismatches.append(
            (
                key,
                tuple(checkpoint_state[key].shape),
                tuple(current_state[key].shape)
            )
        )

print("\n" + "=" * 70)
print("SHARED TENSOR SHAPE VALIDATION")
print("=" * 70)

print(f"Shared tensors:       {len(shared_keys):,}")
print(f"Shape mismatches:     {len(shape_mismatches):,}")

if shape_mismatches:
    print("\nShape mismatches:")
    for key, checkpoint_shape, current_shape in shape_mismatches[:20]:
        print(
            f"  {key}: "
            f"checkpoint={checkpoint_shape}, "
            f"current={current_shape}"
        )

print("\n" + "=" * 70)
print("CHECKPOINT WEIGHT VALIDATION COMPLETED")
print("=" * 70)

CHECKPOINT WEIGHT INTEGRITY VALIDATION

Checkpoint path: ./bart_cnn_dailymail/checkpoint-1000
Weights file exists: True
State dict tensors: 260

KEY WEIGHT VALIDATION

model.encoder.embed_tokens.weight
Present in checkpoint: False

model.decoder.embed_tokens.weight
Present in checkpoint: False

lm_head.weight
Present in checkpoint: False

STATE DICT KEY COMPARISON
Current model tensors:       263
Checkpoint tensors:          260
Missing from checkpoint:     3
Extra checkpoint tensors:    0

Missing keys:
  lm_head.weight
  model.decoder.embed_tokens.weight
  model.encoder.embed_tokens.weight

SHARED TENSOR SHAPE VALIDATION
Shared tensors:       260
Shape mismatches:     0

CHECKPOINT WEIGHT VALIDATION COMPLETED


In [35]:
import os
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("=" * 70)
print("FRESH CHECKPOINT RELOAD VALIDATION")
print("=" * 70)

CHECKPOINT_PATH = "./bart_cnn_dailymail/checkpoint-1000"

if not os.path.isdir(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint directory was not found: {CHECKPOINT_PATH}"
    )

print(f"\nCheckpoint: {CHECKPOINT_PATH}")

# Load the tokenizer directly from the checkpoint.
reload_tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT_PATH
)

# Load the fine-tuned model directly from the checkpoint.
reload_model = AutoModelForSeq2SeqLM.from_pretrained(
    CHECKPOINT_PATH
)

reload_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

reload_model = reload_model.to(reload_device)
reload_model.eval()

print("\n" + "=" * 70)
print("RELOADED MODEL INFORMATION")
print("=" * 70)

reload_total_params = sum(
    p.numel() for p in reload_model.parameters()
)

reload_trainable_params = sum(
    p.numel()
    for p in reload_model.parameters()
    if p.requires_grad
)

print(f"Model class:             {reload_model.__class__.__name__}")
print(f"Tokenizer class:         {reload_tokenizer.__class__.__name__}")
print(f"Device:                  {reload_device}")
print(f"Total parameters:        {reload_total_params:,}")
print(f"Trainable parameters:    {reload_trainable_params:,}")
print(f"Tokenizer vocabulary:    {reload_tokenizer.vocab_size:,}")

# Validate parameter count against the original BART model.
expected_params = 139_420_416

print("\n" + "=" * 70)
print("PARAMETER COUNT VALIDATION")
print("=" * 70)

print(f"Expected BART parameters: {expected_params:,}")
print(f"Reloaded model parameters: {reload_total_params:,}")

if reload_total_params != expected_params:
    raise ValueError(
        "Reloaded model parameter count does not match "
        "the expected BART-base architecture."
    )

print("Parameter count validation: PASSED")

# Validate that the checkpoint can actually generate a summary.
example = tokenized_test[0]

input_ids = torch.tensor(
    [example["input_ids"]],
    dtype=torch.long,
    device=reload_device
)

attention_mask = torch.tensor(
    [example["attention_mask"]],
    dtype=torch.long,
    device=reload_device
)

with torch.no_grad():
    generated_ids = reload_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
        early_stopping=True
    )

reloaded_summary = reload_tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print("\n" + "=" * 70)
print("RELOADED MODEL INFERENCE")
print("=" * 70)

print("\nGenerated summary:")
print(reloaded_summary)

if not reloaded_summary.strip():
    raise ValueError(
        "Reloaded checkpoint generated an empty summary."
    )

print("\n" + "=" * 70)
print("FRESH CHECKPOINT RELOAD VALIDATION PASSED")
print("=" * 70)

FRESH CHECKPOINT RELOAD VALIDATION

Checkpoint: ./bart_cnn_dailymail/checkpoint-1000


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


RELOADED MODEL INFORMATION
Model class:             BartForConditionalGeneration
Tokenizer class:         RobertaTokenizer
Device:                  cuda
Total parameters:        139,420,416
Trainable parameters:    139,420,416
Tokenizer vocabulary:    50,265

PARAMETER COUNT VALIDATION
Expected BART parameters: 139,420,416
Reloaded model parameters: 139,420,416
Parameter count validation: PASSED

RELOADED MODEL INFERENCE

Generated summary:
The revolution in attitudes of everyday Americans has risen 11 points in the past few years alone .
I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world .
The revolution is burning white hot among young people, but also among parents and grandparents .

FRESH CHECKPOINT RELOAD VALIDATION PASSED


In [37]:
# ============================================================
# IMPORTS FOR GENERATION OPTIMIZATION
# ============================================================

import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import evaluate

from transformers import BartForConditionalGeneration, BartTokenizer

print("All required imports loaded successfully.")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

All required imports loaded successfully.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [38]:
# ============================================================
# GENERATION OPTIMIZATION — VALIDATION SET
# ============================================================

EVAL_DATASET = tokenized_val
BATCH_SIZE = 8

generation_configs = {
    "current": {
        "num_beams": 4,
        "length_penalty": 1.0,
        "no_repeat_ngram_size": 0,
    },
    "length_penalty_0.8": {
        "num_beams": 4,
        "length_penalty": 0.8,
        "no_repeat_ngram_size": 0,
    },
    "no_repeat_3": {
        "num_beams": 4,
        "length_penalty": 1.0,
        "no_repeat_ngram_size": 3,
    },
    "length_0.8_no_repeat_3": {
        "num_beams": 4,
        "length_penalty": 0.8,
        "no_repeat_ngram_size": 3,
    },
}

rouge = evaluate.load("rouge")

# ------------------------------------------------------------
# Load final checkpoint
# ------------------------------------------------------------

CHECKPOINT_PATH = "./bart_cnn_dailymail/checkpoint-1000"

generation_model = BartForConditionalGeneration.from_pretrained(
    CHECKPOINT_PATH
).to("cuda")

generation_tokenizer = BartTokenizer.from_pretrained(
    CHECKPOINT_PATH
)

generation_model.eval()

print("=" * 70)
print("GENERATION OPTIMIZATION SETUP")
print("=" * 70)
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Validation examples: {len(EVAL_DATASET)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Configurations: {len(generation_configs)}")
print(f"Device: {next(generation_model.parameters()).device}")
print("=" * 70)


# ------------------------------------------------------------
# Generation function
# ------------------------------------------------------------

def generate_validation_predictions(config):

    predictions = []
    references = []

    for start_idx in tqdm(
        range(0, len(EVAL_DATASET), BATCH_SIZE),
        desc="Generating"
    ):
        end_idx = min(start_idx + BATCH_SIZE, len(EVAL_DATASET))

        batch_examples = [
            EVAL_DATASET[i]
            for i in range(start_idx, end_idx)
        ]

        batch = data_collator(batch_examples)

        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")

        with torch.inference_mode():

            generated_ids = generation_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=config["num_beams"],
                length_penalty=config["length_penalty"],
                no_repeat_ngram_size=config["no_repeat_ngram_size"],
                early_stopping=True,
            )

        decoded_predictions = generation_tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        labels = batch["labels"].clone()
        labels[labels == -100] = generation_tokenizer.pad_token_id

        decoded_references = generation_tokenizer.batch_decode(
            labels,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        predictions.extend(
            [pred.strip() for pred in decoded_predictions]
        )

        references.extend(
            [ref.strip() for ref in decoded_references]
        )

    return predictions, references


# ------------------------------------------------------------
# Run all configurations
# ------------------------------------------------------------

results = []

for config_name, config in generation_configs.items():

    print("\n" + "=" * 70)
    print(f"CONFIGURATION: {config_name}")
    print("=" * 70)

    predictions, references = generate_validation_predictions(config)

    scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    result = {
        "configuration": config_name,
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
        "rougeLsum": scores["rougeLsum"],
        "num_examples": len(predictions),
        "num_beams": config["num_beams"],
        "length_penalty": config["length_penalty"],
        "no_repeat_ngram_size": config["no_repeat_ngram_size"],
    }

    results.append(result)

    print(f"ROUGE-1:     {scores['rouge1']:.6f}")
    print(f"ROUGE-2:     {scores['rouge2']:.6f}")
    print(f"ROUGE-L:     {scores['rougeL']:.6f}")
    print(f"ROUGE-Lsum:  {scores['rougeLsum']:.6f}")


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

generation_results = pd.DataFrame(results)

generation_results = generation_results.sort_values(
    by="rougeLsum",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("GENERATION OPTIMIZATION RESULTS")
print("=" * 70)

display(
    generation_results.style.format({
        "rouge1": "{:.6f}",
        "rouge2": "{:.6f}",
        "rougeL": "{:.6f}",
        "rougeLsum": "{:.6f}",
    })
)

best_generation_config = generation_results.iloc[0].to_dict()

print("\n" + "=" * 70)
print("BEST VALIDATION CONFIGURATION")
print("=" * 70)

print(f"Configuration:       {best_generation_config['configuration']}")
print(f"ROUGE-1:             {best_generation_config['rouge1']:.6f}")
print(f"ROUGE-2:             {best_generation_config['rouge2']:.6f}")
print(f"ROUGE-L:             {best_generation_config['rougeL']:.6f}")
print(f"ROUGE-Lsum:          {best_generation_config['rougeLsum']:.6f}")
print(f"Num beams:           {best_generation_config['num_beams']}")
print(f"Length penalty:      {best_generation_config['length_penalty']}")
print(f"No-repeat ngram:     {best_generation_config['no_repeat_ngram_size']}")
print(f"Validation examples: {best_generation_config['num_examples']}")

print("\nGeneration optimization screening completed successfully.")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

GENERATION OPTIMIZATION SETUP
Checkpoint: ./bart_cnn_dailymail/checkpoint-1000
Validation examples: 1000
Batch size: 8
Configurations: 4
Device: cuda:0

CONFIGURATION: current


Generating:   0%|          | 0/125 [00:00<?, ?it/s]

ROUGE-1:     0.416761
ROUGE-2:     0.197070
ROUGE-L:     0.291146
ROUGE-Lsum:  0.386879

CONFIGURATION: length_penalty_0.8


Generating:   0%|          | 0/125 [00:00<?, ?it/s]

...

ROUGE-1:     0.425967
ROUGE-2:     0.201923
ROUGE-L:     0.295517
ROUGE-Lsum:  0.395292

CONFIGURATION: length_0.8_no_repeat_3


Generating:   0%|          | 0/125 [00:00<?, ?it/s]

ROUGE-1:     0.423454
ROUGE-2:     0.201031
ROUGE-L:     0.295305
ROUGE-Lsum:  0.392590

GENERATION OPTIMIZATION RESULTS


,configuration,rouge1,rouge2,rougeL,rougeLsum,num_examples,num_beams,length_penalty,no_repeat_ngram_size
0,no_repeat_3,0.425967,0.201923,0.295517,0.395292,1000,4,1.000000,3
1,length_0.8_no_repeat_3,0.423454,0.201031,0.295305,0.392590,1000,4,0.800000,3
2,current,0.416761,0.197070,0.291146,0.386879,1000,4,1.000000,0
3,length_penalty_0.8,0.416082,0.196514,0.291269,0.386015,1000,4,0.800000,0



BEST VALIDATION CONFIGURATION
Configuration:       no_repeat_3
ROUGE-1:             0.425967
ROUGE-2:             0.201923
ROUGE-L:             0.295517
ROUGE-Lsum:          0.395292
Num beams:           4
Length penalty:      1.0
No-repeat ngram:     3
Validation examples: 1000

Generation optimization screening completed successfully.


In [39]:
# ============================================================
# FINAL TEST EVALUATION — BEST GENERATION CONFIGURATION
# ============================================================

import time
import torch
import pandas as pd
import evaluate
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration selected from validation
# ------------------------------------------------------------

BEST_GENERATION_CONFIG = {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 3,
    "max_length": 128,
    "early_stopping": True,
}

TEST_DATASET = tokenized_test
BATCH_SIZE = 8

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)
print(f"Test examples:       {len(TEST_DATASET)}")
print(f"Batch size:           {BATCH_SIZE}")
print(f"Num beams:            {BEST_GENERATION_CONFIG['num_beams']}")
print(f"Length penalty:       {BEST_GENERATION_CONFIG['length_penalty']}")
print(f"No-repeat ngram:      {BEST_GENERATION_CONFIG['no_repeat_ngram_size']}")
print(f"Max length:           {BEST_GENERATION_CONFIG['max_length']}")
print("=" * 70)


# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

test_predictions = []
test_references = []

start_time = time.time()

for start_idx in tqdm(
    range(0, len(TEST_DATASET), BATCH_SIZE),
    desc="Final test generation"
):
    end_idx = min(start_idx + BATCH_SIZE, len(TEST_DATASET))

    batch_examples = [
        TEST_DATASET[i]
        for i in range(start_idx, end_idx)
    ]

    batch = data_collator(batch_examples)

    input_ids = batch["input_ids"].to("cuda")
    attention_mask = batch["attention_mask"].to("cuda")

    with torch.inference_mode():

        generated_ids = generation_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=BEST_GENERATION_CONFIG["max_length"],
            num_beams=BEST_GENERATION_CONFIG["num_beams"],
            length_penalty=BEST_GENERATION_CONFIG["length_penalty"],
            no_repeat_ngram_size=BEST_GENERATION_CONFIG["no_repeat_ngram_size"],
            early_stopping=BEST_GENERATION_CONFIG["early_stopping"],
        )

    decoded_predictions = generation_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    labels = batch["labels"].clone()
    labels[labels == -100] = generation_tokenizer.pad_token_id

    decoded_references = generation_tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    test_predictions.extend(
        [pred.strip() for pred in decoded_predictions]
    )

    test_references.extend(
        [ref.strip() for ref in decoded_references]
    )

elapsed_time = time.time() - start_time


# ------------------------------------------------------------
# Calculate final ROUGE
# ------------------------------------------------------------

rouge = evaluate.load("rouge")

final_test_rouge = rouge.compute(
    predictions=test_predictions,
    references=test_references,
    use_stemmer=True
)


# ------------------------------------------------------------
# Previous test results
# ------------------------------------------------------------

previous_test_rouge = {
    "rouge1": 0.406423,
    "rouge2": 0.180911,
    "rougeL": 0.276005,
    "rougeLsum": 0.374562,
}


# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "ROUGE-Lsum",
    ],
    "Previous": [
        previous_test_rouge["rouge1"],
        previous_test_rouge["rouge2"],
        previous_test_rouge["rougeL"],
        previous_test_rouge["rougeLsum"],
    ],
    "Optimized": [
        final_test_rouge["rouge1"],
        final_test_rouge["rouge2"],
        final_test_rouge["rougeL"],
        final_test_rouge["rougeLsum"],
    ],
})

comparison["Absolute Improvement"] = (
    comparison["Optimized"] - comparison["Previous"]
)

comparison["Relative Improvement (%)"] = (
    comparison["Absolute Improvement"]
    / comparison["Previous"]
    * 100
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST ROUGE")
print("=" * 70)

print(f"ROUGE-1:     {final_test_rouge['rouge1']:.6f}")
print(f"ROUGE-2:     {final_test_rouge['rouge2']:.6f}")
print(f"ROUGE-L:     {final_test_rouge['rougeL']:.6f}")
print(f"ROUGE-Lsum:  {final_test_rouge['rougeLsum']:.6f}")

print("\n" + "=" * 70)
print("BEFORE vs OPTIMIZED")
print("=" * 70)

display(
    comparison.style.format({
        "Previous": "{:.6f}",
        "Optimized": "{:.6f}",
        "Absolute Improvement": "{:+.6f}",
        "Relative Improvement (%)": "{:+.2f}%",
    })
)

print("\n" + "=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)

print(f"Test examples evaluated: {len(test_predictions)}")
print(f"Generation time:         {elapsed_time / 60:.2f} minutes")
print(f"Best config:              no_repeat_ngram_size=3")

print("\nFinal test evaluation completed successfully.")

FINAL TEST EVALUATION
Test examples:       1000
Batch size:           8
Num beams:            4
Length penalty:       1.0
No-repeat ngram:      3
Max length:           128


Final test generation:   0%|          | 0/125 [00:00<?, ?it/s]


FINAL TEST ROUGE
ROUGE-1:     0.407043
ROUGE-2:     0.181450
ROUGE-L:     0.276308
ROUGE-Lsum:  0.374985

BEFORE vs OPTIMIZED


,Metric,Previous,Optimized,Absolute Improvement,Relative Improvement (%)
0,ROUGE-1,0.406423,0.407043,+0.000620,+0.15%
1,ROUGE-2,0.180911,0.181450,+0.000539,+0.30%
2,ROUGE-L,0.276005,0.276308,+0.000303,+0.11%
3,ROUGE-Lsum,0.374562,0.374985,+0.000423,+0.11%



EVALUATION SUMMARY
Test examples evaluated: 1000
Generation time:         9.03 minutes
Best config:              no_repeat_ngram_size=3

Final test evaluation completed successfully.


In [40]:
# ============================================================
# FINAL MODEL CONFIGURATION
# ============================================================

import json
import os
from pathlib import Path

FINAL_PACKAGE_DIR = Path("./bart_cnn_dailymail_final")

FINAL_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Final generation configuration
# ------------------------------------------------------------

final_generation_config = {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 3,
    "max_length": 128,
    "early_stopping": True
}

# ------------------------------------------------------------
# Final evaluation metrics
# ------------------------------------------------------------

final_metrics = {
    "rouge1": float(final_test_rouge["rouge1"]),
    "rouge2": float(final_test_rouge["rouge2"]),
    "rougeL": float(final_test_rouge["rougeL"]),
    "rougeLsum": float(final_test_rouge["rougeLsum"])
}

# ------------------------------------------------------------
# Model metadata
# ------------------------------------------------------------

final_metadata = {
    "model_name": "facebook/bart-base",
    "task": "Abstractive Text Summarization",
    "dataset": "abisee/cnn_dailymail",
    "dataset_config": "3.0.0",
    "train_examples": 8000,
    "validation_examples": 1000,
    "test_examples": 1000,
    "max_input_length": 1024,
    "max_target_length": 128,
    "training_epochs": 2,
    "learning_rate": 5e-5,
    "train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "seed": 42,
    "checkpoint": "checkpoint-1000",
    "generation_config": final_generation_config,
    "evaluation_metrics": final_metrics
}

# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------

metadata_path = FINAL_PACKAGE_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        final_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Save generation configuration
# ------------------------------------------------------------

generation_config_path = FINAL_PACKAGE_DIR / "generation_config.json"

with open(generation_config_path, "w", encoding="utf-8") as f:
    json.dump(
        final_generation_config,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Save final metrics
# ------------------------------------------------------------

metrics_path = FINAL_PACKAGE_DIR / "metrics.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        final_metrics,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

print("=" * 70)
print("FINAL PACKAGE METADATA CREATED")
print("=" * 70)

for path in [
    metadata_path,
    generation_config_path,
    metrics_path
]:
    print(f"{path.name:<30} {path.stat().st_size / 1024:.2f} KB")

print("\nFinal ROUGE metrics:")
for metric, value in final_metrics.items():
    print(f"{metric:<12}: {value:.6f}")

print("\nFinal generation configuration:")
for key, value in final_generation_config.items():
    print(f"{key:<24}: {value}")

print("\nMetadata package created successfully.")

FINAL PACKAGE METADATA CREATED
metadata.json                  0.76 KB
generation_config.json         0.12 KB
metrics.json                   0.13 KB

Final ROUGE metrics:
rouge1      : 0.407043
rouge2      : 0.181450
rougeL      : 0.276308
rougeLsum   : 0.374985

Final generation configuration:
num_beams               : 4
length_penalty          : 1.0
no_repeat_ngram_size    : 3
max_length              : 128
early_stopping          : True

Metadata package created successfully.


In [43]:
# ============================================================
# FINAL PACKAGING — CORRECTED
# ============================================================

import shutil
import torch
from pathlib import Path
from transformers import BartForConditionalGeneration, BartTokenizer

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CHECKPOINT_PATH = Path("./bart_cnn_dailymail/checkpoint-1000")
FINAL_PACKAGE_DIR = Path("./bart_cnn_dailymail_final")
MODEL_PACKAGE_DIR = FINAL_PACKAGE_DIR / "model"

MODEL_PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Load validated checkpoint
# ------------------------------------------------------------

print("=" * 70)
print("LOADING VALIDATED CHECKPOINT")
print("=" * 70)

final_model = BartForConditionalGeneration.from_pretrained(
    CHECKPOINT_PATH
)

final_tokenizer = BartTokenizer.from_pretrained(
    CHECKPOINT_PATH
)

final_model.eval()

print(f"Model class:      {final_model.__class__.__name__}")
print(f"Tokenizer class:  {final_tokenizer.__class__.__name__}")
print(f"Parameters:       {sum(p.numel() for p in final_model.parameters()):,}")
print(f"Tokenizer vocab:  {len(final_tokenizer):,}")

# ------------------------------------------------------------
# Save model + tokenizer
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAVING MODEL + TOKENIZER")
print("=" * 70)

final_model.save_pretrained(
    MODEL_PACKAGE_DIR,
    safe_serialization=True
)

final_tokenizer.save_pretrained(
    MODEL_PACKAGE_DIR
)

print(f"Saved to: {MODEL_PACKAGE_DIR}")

# ------------------------------------------------------------
# List package files
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PACKAGED MODEL FILES")
print("=" * 70)

for file_path in sorted(MODEL_PACKAGE_DIR.iterdir()):
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"{file_path.name:<35} {size_mb:>8.2f} MB")

# ------------------------------------------------------------
# Fresh reload from packaged model
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FRESH RELOAD VALIDATION")
print("=" * 70)

packaged_model = BartForConditionalGeneration.from_pretrained(
    MODEL_PACKAGE_DIR
).to("cuda")

packaged_tokenizer = BartTokenizer.from_pretrained(
    MODEL_PACKAGE_DIR
)

packaged_model.eval()

loaded_params = sum(
    p.numel()
    for p in packaged_model.parameters()
)

print(f"Reloaded model:   {packaged_model.__class__.__name__}")
print(f"Reloaded tokenizer: {packaged_tokenizer.__class__.__name__}")
print(f"Parameters:        {loaded_params:,}")
print(f"Device:            {next(packaged_model.parameters()).device}")
print(f"Tokenizer vocab:   {len(packaged_tokenizer):,}")

# ------------------------------------------------------------
# Real inference validation
# ------------------------------------------------------------

test_article = test[0]["article"] if "test" in globals() else None

if test_article is None:
    test_article = generation_tokenizer.decode(
        tokenized_test[0]["input_ids"],
        skip_special_tokens=True
    )

inputs = packaged_tokenizer(
    test_article,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

inputs = {
    key: value.to("cuda")
    for key, value in inputs.items()
}

with torch.inference_mode():

    generated_ids = packaged_model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

generated_summary = packaged_tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

print("\n" + "=" * 70)
print("REAL INFERENCE VALIDATION")
print("=" * 70)

print("Generated summary:")
print("-" * 70)
print(generated_summary)
print("-" * 70)

# ------------------------------------------------------------
# Package size
# ------------------------------------------------------------

total_size = sum(
    p.stat().st_size
    for p in FINAL_PACKAGE_DIR.rglob("*")
    if p.is_file()
)

print("\n" + "=" * 70)
print("FINAL PACKAGE VALIDATION")
print("=" * 70)

print(f"Package directory: {FINAL_PACKAGE_DIR}")
print(f"Total package size: {total_size / (1024 ** 2):.2f} MB")

print("\nFinal package reload + inference validation completed successfully.")

LOADING VALIDATED CHECKPOINT


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Model class:      BartForConditionalGeneration
Tokenizer class:  RobertaTokenizer
Parameters:       139,420,416
Tokenizer vocab:  50,265

SAVING MODEL + TOKENIZER


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: bart_cnn_dailymail_final/model

PACKAGED MODEL FILES
config.json                             0.00 MB
generation_config.json                  0.00 MB
model.safetensors                     532.07 MB
tokenizer.json                          3.39 MB
tokenizer_config.json                   0.00 MB

FRESH RELOAD VALIDATION


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Reloaded model:   BartForConditionalGeneration
Reloaded tokenizer: RobertaTokenizer
Parameters:        139,420,416
Device:            cuda:0
Tokenizer vocab:   50,265

REAL INFERENCE VALIDATION
Generated summary:
----------------------------------------------------------------------
The revolution in attitudes of everyday Americans has risen 11 points in the past few years alone .
I see the revolution in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world .
The revolution is burning white hot among young people, but also among parents and grandparents .
----------------------------------------------------------------------

FINAL PACKAGE VALIDATION
Package directory: bart_cnn_dailymail_final
Total package size: 535.46 MB

Final package reload + inference validation completed successfully.


In [44]:
# ============================================================
# CREATE FINAL PORTABLE ZIP PACKAGE
# ============================================================

import shutil
from pathlib import Path

FINAL_PACKAGE_DIR = Path("./bart_cnn_dailymail_final")
ZIP_BASE = Path("./bart_cnn_dailymail_final")

# ------------------------------------------------------------
# Validate required package structure
# ------------------------------------------------------------

required_paths = [
    FINAL_PACKAGE_DIR / "model",
    FINAL_PACKAGE_DIR / "metadata.json",
    FINAL_PACKAGE_DIR / "metrics.json",
    FINAL_PACKAGE_DIR / "generation_config.json",
]

print("=" * 70)
print("FINAL PACKAGE STRUCTURE VALIDATION")
print("=" * 70)

missing_paths = []

for path in required_paths:
    if path.exists():
        print(f"[OK] {path}")
    else:
        missing_paths.append(str(path))
        print(f"[MISSING] {path}")

if missing_paths:
    raise FileNotFoundError(
        f"Required package items are missing: {missing_paths}"
    )

# ------------------------------------------------------------
# Create ZIP archive
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CREATING ZIP ARCHIVE")
print("=" * 70)

zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=".",
    base_dir=FINAL_PACKAGE_DIR.name
)

zip_file = Path(zip_path)

print(f"ZIP created: {zip_file}")
print(f"ZIP size:    {zip_file.stat().st_size / (1024 ** 2):.2f} MB")

# ------------------------------------------------------------
# Verify ZIP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ZIP VALIDATION")
print("=" * 70)

if zip_file.exists():
    print("[OK] ZIP archive exists")
else:
    raise FileNotFoundError(
        "ZIP archive was not created."
    )

print("\nFinal portable package created successfully.")
print(f"\nDownload path: {zip_file}")

FINAL PACKAGE STRUCTURE VALIDATION
[OK] bart_cnn_dailymail_final/model
[OK] bart_cnn_dailymail_final/metadata.json
[OK] bart_cnn_dailymail_final/metrics.json
[OK] bart_cnn_dailymail_final/generation_config.json

CREATING ZIP ARCHIVE
ZIP created: /content/bart_cnn_dailymail_final.zip
ZIP size:    493.50 MB

ZIP VALIDATION
[OK] ZIP archive exists

Final portable package created successfully.

Download path: /content/bart_cnn_dailymail_final.zip


In [45]:
# ============================================================
# FINAL NOTEBOOK SAVE
# ============================================================

from google.colab import drive
from pathlib import Path

NOTEBOOK_NAME = "Task_7_Text_Summarization_BART_CNN_DailyMail.ipynb"

print("=" * 70)
print("FINAL NOTEBOOK SAVE")
print("=" * 70)

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# Locate current notebook
# ------------------------------------------------------------

notebook_files = list(
    Path("/content").glob("*.ipynb")
)

print("\nNotebook files found:")

for notebook in notebook_files:
    print(f" - {notebook}")

# ------------------------------------------------------------
# Save a copy to Drive
# ------------------------------------------------------------

drive_project_dir = Path(
    "/content/drive/MyDrive/AI_Portfolio/Task_7_Text_Summarization"
)

drive_project_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(f"\nTarget directory:")
print(drive_project_dir)

if notebook_files:

    current_notebook = notebook_files[0]

    target_notebook = drive_project_dir / NOTEBOOK_NAME

    import shutil

    shutil.copy2(
        current_notebook,
        target_notebook
    )

    print("\n" + "=" * 70)
    print("NOTEBOOK SAVED")
    print("=" * 70)

    print(f"Source: {current_notebook}")
    print(f"Saved as: {target_notebook}")

else:

    print("\nNo .ipynb file was found in /content.")
    print("Please save the Colab notebook manually using:")
    print("File > Save a copy in Drive")

FINAL NOTEBOOK SAVE
Mounted at /content/drive

Notebook files found:

Target directory:
/content/drive/MyDrive/AI_Portfolio/Task_7_Text_Summarization

No .ipynb file was found in /content.
Please save the Colab notebook manually using:
File > Save a copy in Drive


In [46]:
# ============================================================
# DOWNLOAD FINAL MODEL PACKAGE
# ============================================================

from google.colab import files
from pathlib import Path

ZIP_PATH = Path("/content/bart_cnn_dailymail_final.zip")

print("=" * 70)
print("FINAL MODEL PACKAGE DOWNLOAD")
print("=" * 70)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP file not found: {ZIP_PATH}"
    )

size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)

print(f"File: {ZIP_PATH.name}")
print(f"Size: {size_mb:.2f} MB")
print("\nStarting browser download...")

files.download(str(ZIP_PATH))

print("\nDownload request sent successfully.")

FINAL MODEL PACKAGE DOWNLOAD
File: bart_cnn_dailymail_final.zip
Size: 493.50 MB

Starting browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download request sent successfully.
